# LeetCode 440 — K-th Smallest in Lexicographical Order (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def findKthNumber(self, n: int, k: int) -> int:

```

**Mục tiêu học:**

1. Hiểu thật chắc **thứ tự từ điển** (lexicographical order) của các số: vì sao `"10"` đứng **trước** `"2"`, vì sao `1` đứng trước `10`.
2. Ôn lại bài anh em **#386 Lexicographical Numbers** (liệt kê `1..n` theo thứ tự từ điển) từ con số 0: bằng `sorted(..., key=str)`, bằng đệ quy DFS, và bằng vòng lặp "nhân 10 / cộng 1 / chia 10".
3. Nhìn các số `1..n` như một **cây 10 nhánh** (10-ary tree, cây tiền tố): nút `x` có các con `10x, 10x+1, ..., 10x+9`. Hiểu vì sao **duyệt pre-order** cây này cho đúng thứ tự từ điển.
4. Viết được 2 cách "dễ nghĩ" — **sort** (`O(n log n)`, làm trọng tài) và **đi bộ pre-order đếm tới `k`** (`O(k)`) — và thấy rõ vì sao cả hai đều TLE khi `n = k = 10^9`.
5. Nắm cách chính: **đếm số nút trong cây con tiền tố** bằng hai biên `first`/`last` nhân 10 mỗi tầng (`count_steps`), rồi **nhảy qua cả cây con** (`prefix += 1`) hoặc **đi xuống** (`prefix *= 10`). Chỉ khoảng vài trăm phép tính cho `n = 10^9`, tức `O(log² n)`.
6. Hiểu vì sao cách chính đúng (invariant), tránh các lỗi off-by-one kinh điển (`k -= 1`, `<=` hay `<`, quên `min(last, n)`).

**Lộ trình 3 cách** (từ cách người mới tự nghĩ ra đến cách tối ưu):

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---|---|
| 1 | Sort `1..n` theo chuỗi (`key=str`), lấy phần tử thứ `k` — **trọng tài** | `O(n log n)` | `O(n)` |
| 2 | Đi pre-order trên cây 10 nhánh (đệ quy hoặc vòng lặp "×10 / +1 / ÷10"), đếm tới `k` | `O(k)` | `O(log n)` / `O(1)` |
| 3 | **Đếm cây con bằng `count_steps`, nhảy qua cả cây con hoặc đi xuống** — cách chính | `O(log² n)` | `O(1)` |

## 1. Đề bài

> Given two integers `n` and `k`, return *the* `k`*-th lexicographically smallest integer in the range* `[1, n]`.

Dịch: Cho hai số nguyên `n` và `k`. Hãy trả về số nguyên **nhỏ thứ `k` theo thứ tự từ điển** trong đoạn `[1, n]` (tức trong các số `1, 2, ..., n`).

**Ví dụ:**

| Input | Output | Giải thích |
|---|---:|---|
| `n = 13, k = 2` | `10` | Thứ tự từ điển là `[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]`, nên số nhỏ thứ 2 là `10`. |
| `n = 1, k = 1` | `1` | Chỉ có đúng một số. |

**Constraints:**

- `1 <= k <= n <= 10^9`

Ba điều cần để ý ngay từ đầu:

- `k` đếm **từ 1** (số đầu tiên là "thứ 1"), còn chỉ số list trong Python đếm **từ 0**. Đây là nguồn gốc của rất nhiều lỗi lệch một (off-by-one).
- Đề luôn đảm bảo `k <= n`, nên đáp án luôn tồn tại.
- `n` và `k` có thể tới **`10^9`** (một tỉ). Con số này loại thẳng mọi cách "liệt kê hết rồi chọn" — kể cả cách chỉ đi `k` bước.

### Hiểu Ví dụ 1 bằng tay (`n = 13`)

"Thứ tự từ điển" nghĩa là: **viết mỗi số thành chuỗi ký tự rồi sắp xếp như sắp xếp từ trong từ điển**.

| Thứ tự `k` | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Số | `1` | **`10`** | `11` | `12` | `13` | `2` | `3` | `4` | `5` | `6` | `7` | `8` | `9` |

Quan sát:

- Mọi số **bắt đầu bằng chữ số `1`** (`1, 10, 11, 12, 13`) dồn thành một khối ở đầu, rồi mới tới `2`.
- Trong khối đó, `1` đứng trước `10` vì `"1"` là **tiền tố** (prefix — đoạn đầu) của `"10"`.
- `13` đứng **trước** `2` dù `13 > 2` về giá trị — vì so ký tự đầu: `'1' < '2'`.

Cell dưới kiểm chứng bằng code. Hàm `sorted(..., key=str)` sẽ được giải thích kỹ ở mục 2.

In [1]:
n = 13
order = sorted(range(1, n + 1), key=str)   # sắp xếp theo dạng CHUỖI của mỗi số
print("Thứ tự từ điển:", order)
for k in [1, 2, 5, 6, 13]:
    print(f"k = {k:>2} -> {order[k - 1]}")   # k đếm từ 1, list đếm từ 0 nên dùng k - 1

Thứ tự từ điển: [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
k =  1 -> 1
k =  2 -> 10
k =  5 -> 13
k =  6 -> 2
k = 13 -> 9


## 2. Kiến thức nền 1 — So sánh chuỗi theo thứ tự từ điển

Python so sánh hai chuỗi `a < b` giống hệt cách tra từ điển:

1. So **ký tự đầu tiên** của hai chuỗi. Ký tự nào "nhỏ hơn" thì chuỗi đó nhỏ hơn — dừng luôn, **không cần nhìn phần sau**.
2. Nếu bằng nhau, so tiếp ký tự thứ hai, thứ ba, ...
3. Nếu một chuỗi **hết ký tự trước** (nó là tiền tố của chuỗi kia) thì chuỗi **ngắn hơn** nhỏ hơn: `"1" < "10"`, giống `"an" < "anh"`.

Các ký tự chữ số được so theo mã (`ord`): `'0' < '1' < ... < '9'` (mã 48 → 57).

Ví dụ từng bước:

| So sánh | Bước quyết định | Kết quả |
|---|---|---|
| `"10"` vs `"2"` | ký tự đầu `'1'` vs `'2'` → `'1'` nhỏ hơn | `"10" < "2"` |
| `"13"` vs `"129"` | `'1'='1'`, rồi `'3'` vs `'2'` → `'2'` nhỏ hơn | `"129" < "13"` |
| `"1"` vs `"10"` | `'1'='1'`, rồi `"1"` hết ký tự | `"1" < "10"` |
| `"100"` vs `"99"` | `'1'` vs `'9'` | `"100" < "99"` |

Còn khi so **số** (`int`) thì Python so theo giá trị: `10 > 2`. Bài này đòi thứ tự **chuỗi**, không phải thứ tự số.

In [2]:
print("10" < "2", 10 < 2)         # True False  : chuỗi và số cho kết quả ngược nhau!
print("129" < "13")               # True        : khác nhau ở ký tự thứ 2, '2' < '3'
print("1" < "10")                 # True        : "1" là tiền tố của "10"
print("100" < "99")               # True        : '1' < '9' ngay ký tự đầu
print(ord("0"), ord("1"), ord("9"))   # 48 49 57 : mã của các ký tự chữ số

True False
True
True
True
48 49 57


## 3. Kiến thức nền 2 — `sorted(..., key=str)` và `str.startswith`

**`sorted(iterable)`** trả về một **list mới** đã sắp xếp tăng dần. Tham số **`key`** là một hàm: Python sẽ sắp xếp theo **`key(x)`** thay vì theo chính `x`, nhưng list kết quả vẫn chứa các `x` gốc.

- `sorted([3, 21, 100, 9])` → `[3, 9, 21, 100]` (theo giá trị số).
- `sorted([3, 21, 100, 9], key=str)` → so `"3", "21", "100", "9"` → `[100, 21, 3, 9]`.

Lưu ý: kết quả vẫn là **số** (`int`), không phải chuỗi. Ta chỉ "mượn" chuỗi để so sánh.

**`s.startswith(t)`** trả về `True` nếu chuỗi `s` **bắt đầu bằng** chuỗi `t`. Ta sẽ dùng nó để nói "số `x` nằm trong nhóm có tiền tố `p`": `str(x).startswith(str(p))`.

- `"123".startswith("12")` → `True`
- `"12".startswith("123")` → `False` (chuỗi ngắn không thể bắt đầu bằng chuỗi dài hơn)

In [3]:
nums = [3, 21, 100, 9]
print(sorted(nums))               # [3, 9, 21, 100]   theo giá trị
print(sorted(nums, key=str))      # [100, 21, 3, 9]   theo chuỗi
print(sorted(map(str, nums)))     # ['100', '21', '3', '9'] : nếu đổi hẳn sang chuỗi thì kết quả là chuỗi

print("123".startswith("12"), "12".startswith("123"))   # True False

# Các số trong [1, 13] có tiền tố "1":
print([x for x in range(1, 14) if str(x).startswith("1")])   # [1, 10, 11, 12, 13]

[3, 9, 21, 100]
[100, 21, 3, 9]
['100', '21', '3', '9']
True False
[1, 10, 11, 12, 13]


## 4. Ôn bài #386 Lexicographical Numbers (Medium) — cách sort

Bài **LeetCode 386** hỏi: *"Cho `n`, trả về **tất cả** các số `1..n` theo thứ tự từ điển"* (đề gốc yêu cầu `O(n)` thời gian và `O(1)` bộ nhớ phụ — ta sẽ đạt được ở mục 9).

Bài 440 của chúng ta chỉ là: **lấy phần tử thứ `k` trong danh sách của bài 386.** Vì vậy mọi kỹ thuật của 386 đều dùng được — chỉ là chậm.

Cách đầu tiên ai cũng nghĩ ra cho 386: `sorted(range(1, n + 1), key=str)`. Đơn giản, đúng, nhưng tốn `O(n log n)` thời gian và phải giữ cả `n` số trong bộ nhớ.

Cell dưới in thử với vài `n` để "cảm" thứ tự từ điển. Để ý `n = 100`: số `100` chen vào giữa `10` và `11`.

In [4]:
def lex_order_sort(n: int) -> list:
    """#386 bằng sort: các số 1..n theo thứ tự từ điển."""
    return sorted(range(1, n + 1), key=str)

print("n =  9:", lex_order_sort(9))
print("n = 13:", lex_order_sort(13))
print("n = 25:", lex_order_sort(25))
print("n = 100, 15 số đầu:", lex_order_sort(100)[:15])

n =  9: [1, 2, 3, 4, 5, 6, 7, 8, 9]
n = 13: [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
n = 25: [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 3, 4, 5, 6, 7, 8, 9]
n = 100, 15 số đầu: [1, 10, 100, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21]


## 5. Kiến thức nền 3 — Cây (tree) và cây 10 nhánh của các tiền tố

**Cây** là cấu trúc gồm các **nút** (node) nối với nhau kiểu "gia phả":

- **Gốc** (root): nút trên cùng, không có cha.
- Mỗi nút (trừ gốc) có đúng một **cha** (parent); một nút có thể có nhiều **con** (children).
- **Lá** (leaf): nút không có con.
- **Cây con** (subtree) của nút `x`: gồm `x` và **toàn bộ con cháu** của `x`.
- **Tầng** (level/depth): khoảng cách từ nút tới gốc.

**Cây tiền tố 10 nhánh của bài này** (10-ary prefix tree):

- Một **gốc ảo** (không phải số nào) có 9 con: `1, 2, ..., 9` (số không bắt đầu bằng `0`).
- Nút `x` có tối đa 10 con: **thêm một chữ số vào cuối** `x`, tức `10x + 0, 10x + 1, ..., 10x + 9`. Ví dụ con của `1` là `10..19`, con của `12` là `120..129`.
- Chỉ giữ các nút `<= n`.

Hệ quả quan trọng: **cha của `x` là `x // 10`** (bỏ chữ số cuối), và **cây con của `p` chứa đúng những số có tiền tố `p`**. Ví dụ với `n = 13`, cây con của `1` là `{1, 10, 11, 12, 13}`.

Hình cây với `n = 13` (in bằng hàm ở cell dưới):

```text
(gốc ảo)
├── 1
│   ├── 10
│   ├── 11
│   ├── 12
│   └── 13
├── 2
├── 3
...
└── 9
```

In [5]:
def children(x: int, n: int) -> list:
    """Các con của nút x trong cây tiền tố (chỉ giữ số <= n)."""
    return [c for c in range(10 * x, 10 * x + 10) if c <= n]

def subtree_size_brute(n: int, p: int) -> int:
    """Số nút trong cây con gốc p = số x trong [1, n] có tiền tố p (đếm vét cạn)."""
    return sum(1 for x in range(1, n + 1) if str(x).startswith(str(p)))

def print_prefix_tree(n: int, expand=None):
    """Vẽ cây tiền tố bằng ASCII. expand = tập nút được bung ra (None = bung hết)."""
    def draw(x, prefix, is_last):
        connector = "└── " if is_last else "├── "
        kids = children(x, n)
        if kids and expand is not None and x not in expand:
            size = subtree_size_brute(n, x)
            print(f"{prefix}{connector}{x}   [thu gọn: cây con {size} nút = {x}, {kids[0]}..{kids[-1]} ...]")
            return
        print(prefix + connector + str(x))
        new_prefix = prefix + ("    " if is_last else "│   ")
        for i, c in enumerate(kids):
            draw(c, new_prefix, i == len(kids) - 1)

    tops = [d for d in range(1, 10) if d <= n]
    print("(gốc ảo)")
    for i, d in enumerate(tops):
        draw(d, "", i == len(tops) - 1)

print("Con của 1 (n=13):", children(1, 13), "| con của 12 (n=1234):", children(12, 1234))
print_prefix_tree(13)

Con của 1 (n=13): [10, 11, 12, 13] | con của 12 (n=1234): [120, 121, 122, 123, 124, 125, 126, 127, 128, 129]
(gốc ảo)
├── 1
│   ├── 10
│   ├── 11
│   ├── 12
│   └── 13
├── 2
├── 3
├── 4
├── 5
├── 6
├── 7
├── 8
└── 9


Với `n = 100`, cây có 100 nút — in hết sẽ dài, nên ta chỉ **bung** nhánh `1` và `10`, các nhánh khác **thu gọn** kèm kích thước cây con:

- Cây con `1` = `{1, 10, 100, 11, 12, ..., 19}` → **12 nút** (số `100` là cháu của `1`, con của `10`).
- Cây con `2` = `{2, 20, ..., 29}` → **11 nút**; tương tự cho `3, ..., 9`.
- Tổng: `12 + 8 · 11 = 100` ✓ — đúng bằng `n`, vì mỗi số `1..n` nằm ở đúng một nhánh.

Nhớ các con số `12` và `11` này — chúng sẽ quay lại trong phần trace của cách chính.

In [6]:
print_prefix_tree(100, expand={1, 10})
sizes = {d: subtree_size_brute(100, d) for d in range(1, 10)}
print("Kích thước cây con các nhánh đầu:", sizes, "| tổng =", sum(sizes.values()))

(gốc ảo)
├── 1
│   ├── 10
│   │   └── 100
│   ├── 11
│   ├── 12
│   ├── 13
│   ├── 14
│   ├── 15
│   ├── 16
│   ├── 17
│   ├── 18
│   └── 19
├── 2   [thu gọn: cây con 11 nút = 2, 20..29 ...]
├── 3   [thu gọn: cây con 11 nút = 3, 30..39 ...]
├── 4   [thu gọn: cây con 11 nút = 4, 40..49 ...]
├── 5   [thu gọn: cây con 11 nút = 5, 50..59 ...]
├── 6   [thu gọn: cây con 11 nút = 6, 60..69 ...]
├── 7   [thu gọn: cây con 11 nút = 7, 70..79 ...]
├── 8   [thu gọn: cây con 11 nút = 8, 80..89 ...]
└── 9   [thu gọn: cây con 11 nút = 9, 90..99 ...]
Kích thước cây con các nhánh đầu: {1: 12, 2: 11, 3: 11, 4: 11, 5: 11, 6: 11, 7: 11, 8: 11, 9: 11} | tổng = 100


## 6. Kiến thức nền 4 — Đệ quy (recursion) và duyệt pre-order

**Đệ quy** là khi một hàm **tự gọi lại chính nó** cho một bài toán con nhỏ hơn. Mỗi hàm đệ quy cần:

1. **Điểm dừng** (base case): khi nào thì không gọi tiếp nữa.
2. **Bước đệ quy**: giải phần việc của mình, rồi giao phần còn lại cho các lời gọi con.

**Duyệt pre-order** (tiền thứ tự) một cây: tại mỗi nút, **thăm nút đó TRƯỚC**, rồi lần lượt duyệt (đệ quy) cây con của từng con **từ trái sang phải**. Viết bằng đệ quy chỉ 3 dòng:

```python
def preorder(x):
    visit(x)              # 1. thăm chính nút x
    for c in con_của(x):  # 2. rồi đi hết từng cây con, theo thứ tự
        preorder(c)
```

Thử với một "gia phả" nhỏ: `A` có con `B, C`; `B` có con `D, E`; `C` có con `F`. Pre-order là **A → B → D → E → C → F**: đi sâu hết nhánh `B` (cả `D`, `E`) rồi mới sang `C`. Điểm then chốt: **toàn bộ cây con của một nút được thăm liền một mạch**, không bị nút khác chen ngang.

In [7]:
family = {"A": ["B", "C"], "B": ["D", "E"], "C": ["F"], "D": [], "E": [], "F": []}

def preorder(node, depth=0, out=None):
    if out is None:
        out = []
    out.append(node)
    print("    " * depth + f"thăm {node} (tầng {depth})")
    for child in family[node]:          # điểm dừng: lá không có con -> vòng for không chạy
        preorder(child, depth + 1, out)
    return out

print("Thứ tự pre-order:", preorder("A"))

thăm A (tầng 0)
    thăm B (tầng 1)
        thăm D (tầng 2)
        thăm E (tầng 2)
    thăm C (tầng 1)
        thăm F (tầng 2)
Thứ tự pre-order: ['A', 'B', 'D', 'E', 'C', 'F']


### Vì sao pre-order của cây tiền tố chính là thứ tự từ điển?

Lấy hai số khác nhau `a`, `b` trong `[1, n]`, viết thành chuỗi. Có đúng 2 khả năng:

1. **`a` là tiền tố của `b`** (ví dụ `a = 1`, `b = 13`). Khi đó `a` là **tổ tiên** của `b` trong cây → pre-order thăm `a` trước (thăm nút trước con cháu). Và theo từ điển, chuỗi ngắn hơn đứng trước: `"1" < "13"`. **Khớp.**
2. **Hai chuỗi khác nhau ở vị trí đầu tiên `i`** (ví dụ `a = 129`, `b = 13`: phần chung `"1"`, rồi `'2'` vs `'3'`). Phần chung `"1"` là nút tổ tiên chung; `a` nằm trong cây con của `12`, `b` nằm trong cây con của `13`. Pre-order duyệt con theo chữ số tăng dần và **thăm trọn cây con `12` rồi mới sang `13`** → `a` trước `b`. Theo từ điển cũng vậy vì `'2' < '3'`. **Khớp.**

Vậy: **thứ tự từ điển của `1..n` = thứ tự pre-order của cây tiền tố.** Bài 440 trở thành: *"đi pre-order, nút thứ `k` là nút nào?"*

Cell dưới kiểm chứng bằng DFS (Depth-First Search — tìm kiếm theo chiều sâu, chính là pre-order bằng đệ quy) cho #386.

In [8]:
def lex_order_dfs(n: int) -> list:
    """#386 bằng DFS pre-order trên cây tiền tố."""
    result = []

    def dfs(x):
        result.append(x)                  # thăm x trước
        for c in range(10 * x, 10 * x + 10):
            if c > n:                     # các con sau còn lớn hơn -> dừng sớm
                break
            dfs(c)

    for first in range(1, 10):            # 9 con của gốc ảo
        if first > n:
            break
        dfs(first)
    return result

print(lex_order_dfs(13))
for n in range(1, 1001):
    assert lex_order_dfs(n) == lex_order_sort(n), n
print("DFS pre-order khớp sorted(key=str) với mọi n trong [1, 1000]")

[1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]


DFS pre-order khớp sorted(key=str) với mọi n trong [1, 1000]


## 7. Cách 1 — Sort theo chuỗi rồi lấy phần tử thứ `k` (dùng làm "trọng tài")

Đây là cách dịch thẳng đề sang code: **tạo danh sách `1..n`, sắp theo `str`, lấy phần tử ở chỉ số `k - 1`.**

Cách này làm đúng nguyên văn định nghĩa nên gần như không thể sai → ta dùng nó làm **trọng tài** (oracle) để kiểm tra các cách nhanh hơn trên `n` nhỏ.

**Độ phức tạp.** Gọi `d` là số chữ số của `n` (`d ≤ 10`).

- Sort `n` phần tử cần `O(n log n)` phép so sánh, mỗi phép so chuỗi tốn tới `O(d)` → **Time `O(n log n · d)`**, thường viết gọn `O(n log n)`.
- Giữ cả list `n` số và `n` chuỗi khóa → **Space `O(n)`**.

In [9]:
def kth_sort(n: int, k: int) -> int:
    return sorted(range(1, n + 1), key=str)[k - 1]

print(kth_sort(13, 2))     # 10 (Ví dụ 1)
print(kth_sort(1, 1))      # 1  (Ví dụ 2)
print(kth_sort(100, 10))   # 17
print(kth_sort(100, 50))   # 53

10
1
17
53


### Cách 1 vướng ở đâu? Đo thời gian và bộ nhớ

Với `n = 10^9`:

- **Bộ nhớ**: một tỉ đối tượng `int` trong Python (mỗi cái ~28 byte, cộng 8 byte con trỏ trong list), cộng thêm một tỉ chuỗi khóa (~50 byte mỗi cái) → **hàng chục GB**. LeetCode cấp vài trăm MB → **MLE** (Memory Limit Exceeded) trước cả khi kịp TLE.
- **Thời gian**: đo với `n = 10^5` và `10^6` rồi ngoại suy. Thời gian tăng hơi nhanh hơn tuyến tính (vì có `log n`), nên nhân 1000 còn là ước lượng **lạc quan**.

In [10]:
import time

for n in [10**5, 10**6]:
    start = time.perf_counter()
    ans = kth_sort(n, n // 2)
    elapsed = time.perf_counter() - start
    print(f"n = {n:>9,}: đáp án k = n/2 là {ans:>7}, mất {elapsed:.3f} giây")

print(f"Ước lượng cho n = 10^9: > {elapsed * 1000:.0f} giây (≈ {elapsed * 1000 / 60:.0f} phút), chưa kể hết RAM")

n =   100,000: đáp án k = n/2 là   54998, mất 0.016 giây


n = 1,000,000: đáp án k = n/2 là  549998, mất 0.170 giây
Ước lượng cho n = 10^9: > 170 giây (≈ 3 phút), chưa kể hết RAM


## 8. Cách 2 — Đi pre-order và dừng ở nút thứ `k` (`O(k)`)

**Cách 1 lãng phí ở đâu?** Nó sắp xếp **cả `n` số**, trong khi ta chỉ cần **`k` số đầu tiên** của thứ tự từ điển. Ví dụ `n = 10^9, k = 3`: đáp án là `100`, chỉ cần đi `1 → 10 → 100`!

Nhờ mục 6, ta biết thứ tự từ điển = pre-order của cây tiền tố. Vậy: **đi pre-order, đếm số nút đã thăm, tới nút thứ `k` thì dừng và trả về nó.** Không sort, không lưu list.

Có hai cách cài đặt:

- **2a — Đệ quy DFS** (y hệt `lex_order_dfs`, thêm bộ đếm và dừng sớm).
- **2b — Vòng lặp "nhân 10 / cộng 1 / chia 10"**: tính trực tiếp "số kế tiếp" theo thứ tự từ điển, không cần đệ quy (mục 9).

Để viết 2a, ta cần một công cụ Python nhỏ: `nonlocal`.

### Kiến thức nền 5 — `nonlocal`: cho hàm con sửa biến của hàm cha

Khi một hàm con (định nghĩa bên trong hàm khác) **gán** `count += 1`, Python mặc định coi `count` là biến **cục bộ mới** của hàm con → báo lỗi `UnboundLocalError` (dùng biến trước khi gán).

Khai báo `nonlocal count` ở đầu hàm con để nói: *"`count` này là biến của hàm bao ngoài, hãy sửa trực tiếp nó."*

(Chỉ **đọc** biến của hàm cha thì không cần `nonlocal`; chỉ khi **gán lại** mới cần. Với list thì `result.append(x)` là sửa *bên trong* list chứ không gán lại tên `result`, nên `lex_order_dfs` ở trên không cần `nonlocal`.)

In [11]:
def counter_ok():
    count = 0
    def bump():
        nonlocal count        # dùng biến count của counter_ok
        count += 1
    bump(); bump(); bump()
    return count

def counter_broken():
    count = 0
    def bump():
        count += 1            # thiếu nonlocal -> Python tưởng count là biến cục bộ của bump
    bump()
    return count

print("Có nonlocal  :", counter_ok())          # 3
try:
    counter_broken()
except UnboundLocalError as e:
    print("Thiếu nonlocal:", type(e).__name__, "-", e)

Có nonlocal  : 3
Thiếu nonlocal: UnboundLocalError - cannot access local variable 'count' where it is not associated with a value


### Cách 2a — DFS đệ quy, đếm tới `k` rồi dừng

- `count` = số nút đã thăm. Mỗi lần thăm một nút: `count += 1`; nếu `count == k` thì ghi lại `answer` và **dừng toàn bộ**.
- Sau mỗi lời gọi con, kiểm tra `answer is not None` để **thoát sớm**, không đi tiếp vô ích.
- Độ sâu đệ quy tối đa = số chữ số của `n` (`≤ 10`), nên không lo tràn stack.
- **Time `O(k)`** (thăm `k` nút, mỗi nút thêm tối đa một lần thử con vượt `n`), **Space `O(log n)`** cho stack đệ quy.

In [12]:
def kth_dfs(n: int, k: int) -> int:
    count = 0
    answer = None

    def dfs(x):
        nonlocal count, answer
        count += 1                         # thăm x: nó là số thứ count
        if count == k:
            answer = x
            return
        for c in range(10 * x, 10 * x + 10):
            if c > n or answer is not None:
                break
            dfs(c)

    for first in range(1, 10):
        if first > n or answer is not None:
            break
        dfs(first)
    return answer

print(kth_dfs(13, 2), kth_dfs(1, 1), kth_dfs(100, 10), kth_dfs(100, 50))   # 10 1 17 53
print(kth_dfs(10**9, 3))   # 100 : chỉ cần thăm 1 -> 10 -> 100, dù n là một tỉ

10 1 17 53
100


## 9. Cách 2b — Vòng lặp "nhân 10 / cộng 1 / chia 10" (lời giải `O(1)` bộ nhớ của #386)

Câu hỏi: đang đứng ở số `cur`, **số kế tiếp** theo thứ tự từ điển là gì? Nhìn cây, pre-order đi theo 3 luật, **ưu tiên từ trên xuống**:

1. **Đi xuống con đầu** nếu được: `cur * 10 <= n` → `cur = cur * 10`. (Ví dụ `n = 13`: `1 → 10`.)
2. Không xuống được thì **sang em kế bên phải**: `cur + 1`. Nhưng chỉ được làm vậy nếu `cur + 1` **thật sự là anh em** của `cur` và vẫn `<= n`:
   - Nếu `cur` tận cùng bằng `9` (ví dụ `19`), thì `cur + 1 = 20` **không** phải em của `19` (nó là con của `2`, khác cha!).
   - Nếu `cur + 1 > n` (ví dụ `n = 13`, `cur = 13`) thì không có em.
3. Gặp hai trường hợp trên thì **lùi lên cha**: `cur = cur // 10`, lặp lại cho tới khi sang phải được, rồi `cur += 1`.

Viết gọn:

```python
if cur * 10 <= n:
    cur *= 10
else:
    while cur % 10 == 9 or cur + 1 > n:
        cur //= 10
    cur += 1
```

Ví dụ tay với `n = 13`:

| `cur` | `cur*10 <= 13`? | Vì sao lùi lên? | Số kế tiếp |
|---:|---|---|---:|
| `1` | `10 <= 13` ✓ | — | `10` |
| `10` | `100` ✗ | không cần lùi (`11 <= 13`, không tận cùng 9) | `11` |
| `13` | `130` ✗ | `14 > 13` → lùi về `1`; `2 <= 13` → dừng lùi | `2` |
| `2` | `20` ✗ | không cần | `3` |

Với `n = 100`: từ `19` → `190 > 100`, `19 % 10 == 9` → lùi về `1` → `+1` → `2`. Từ `100` → `1000 > 100`, `101 > 100` → lùi về `10` → `+1` → `11`. Với `n = 25`: từ `25` → `250 > 25`, `26 > 25` → lùi về `2` → `+1` → `3`.

(Lưu ý: `next_lex` chỉ gọi khi `cur` **chưa phải số cuối** theo từ điển — với số cuối như `9` khi `n = 13` thì không còn "số kế tiếp" nào.)

In [13]:
def next_lex(cur: int, n: int) -> int:
    """Số đứng ngay sau cur theo thứ tự từ điển trong [1, n] (giả sử cur chưa phải số cuối)."""
    if cur * 10 <= n:
        return cur * 10                    # luật 1: xuống con đầu
    while cur % 10 == 9 or cur + 1 > n:    # luật 3: hết em bên phải -> lùi lên cha
        cur //= 10
    return cur + 1                         # luật 2: sang em kế bên phải

for cur, n in [(1, 13), (10, 13), (13, 13), (2, 13), (19, 100), (100, 100), (25, 25)]:
    print(f"n = {n:>3}: sau {cur:>3} là {next_lex(cur, n)}")

n =  13: sau   1 là 10
n =  13: sau  10 là 11
n =  13: sau  13 là 2
n =  13: sau   2 là 3
n = 100: sau  19 là 2
n = 100: sau 100 là 11
n =  25: sau  25 là 3


Với `next_lex` trong tay:

- **#386** (liệt kê hết): bắt đầu `cur = 1`, gọi `next_lex` `n - 1` lần. Không đệ quy, không list phụ ngoài kết quả → đúng yêu cầu `O(n)` thời gian, `O(1)` bộ nhớ phụ của #386.
- **#440 bằng Cách 2b**: bắt đầu `cur = 1` (là số thứ 1), gọi `next_lex` **`k - 1` lần** → `cur` là số thứ `k`.

Hàm `trace_kth_walk` in từng bước để thấy rõ luật nào được áp dụng. Ta chạy với `n = 13, k = 7` (đáp án `3`) để thấy cả ba luật: xuống, sang phải, và lùi lên.

In [14]:
def lex_order_iter(n: int) -> list:
    """#386 không đệ quy, O(1) bộ nhớ phụ."""
    result = [1]
    cur = 1
    for _ in range(n - 1):
        cur = next_lex(cur, n)
        result.append(cur)
    return result

def kth_walk(n: int, k: int) -> int:
    cur = 1                        # đang đứng ở số thứ 1
    for _ in range(k - 1):         # đi thêm k - 1 bước
        cur = next_lex(cur, n)
    return cur

def trace_kth_walk(n, k):
    cur = 1
    print(f"Bước 1: cur = {cur}")
    for step in range(2, k + 1):
        if cur * 10 <= n:
            how = f"xuống con đầu ({cur}*10)"
        elif cur % 10 == 9 or cur + 1 > n:
            how = "lùi lên cha rồi sang phải"
        else:
            how = f"sang em bên phải ({cur}+1)"
        cur = next_lex(cur, n)
        print(f"Bước {step}: cur = {cur:<3} <- {how}")
    return cur

for n in range(1, 1001):
    assert lex_order_iter(n) == lex_order_sort(n), n
print("lex_order_iter khớp sort với mọi n trong [1, 1000]\n")
print("Đáp án:", trace_kth_walk(13, 7))

lex_order_iter khớp sort với mọi n trong [1, 1000]

Bước 1: cur = 1
Bước 2: cur = 10  <- xuống con đầu (1*10)
Bước 3: cur = 11  <- sang em bên phải (10+1)
Bước 4: cur = 12  <- sang em bên phải (11+1)
Bước 5: cur = 13  <- sang em bên phải (12+1)
Bước 6: cur = 2   <- lùi lên cha rồi sang phải
Bước 7: cur = 3   <- sang em bên phải (2+1)
Đáp án: 3


### Cách 2 vướng ở đâu?

Cả 2a và 2b đều đi **đúng `k` bước** → **Time `O(k)`**. Bộ nhớ phụ: 2a là `O(log n)` (stack đệ quy), 2b là `O(1)`. (Về `next_lex`: một bước đơn lẻ có thể lùi lên nhiều tầng, nhưng mỗi lần lùi là "đóng" một nút đã thăm, nên tổng số lần lùi trong `k` bước không quá `k + log n` → tổng vẫn `O(k)`.)

So với Cách 1 đây là bước tiến lớn: không cần bộ nhớ `O(n)`, và nếu `k` nhỏ thì cực nhanh dù `n` là một tỉ. Nhưng **đề cho `k` tới `10^9`**. Ví dụ `n = k = 10^9` (đáp án `999999999`, số cuối cùng): phải đi một tỉ bước. Đo với `k = 10^6` rồi nhân 1000:

In [15]:
start = time.perf_counter()
ans = kth_walk(10**9, 10**6)
elapsed = time.perf_counter() - start
print(f"kth_walk(10^9, 10^6) = {ans}, mất {elapsed:.3f} giây")
print(f"Ước lượng cho k = 10^9: khoảng {elapsed * 1000:.0f} giây (≈ {elapsed * 1000 / 60:.0f} phút) -> TLE")

kth_walk(10^9, 10^6) = 100899996, mất 0.157 giây
Ước lượng cho k = 10^9: khoảng 157 giây (≈ 3 phút) -> TLE


## 10. Bước đột phá — Đừng đi bộ từng nút: **nhảy qua cả cây con**

Hãy tưởng tượng tìm **trang thứ 500** trong một cuốn sách mà bạn chỉ được lật từng trang (Cách 2). Chậm! Cách thông minh: nhìn **mục lục** — "Chương 1 dài 120 trang, Chương 2 dài 150 trang, Chương 3 dài 300 trang". Trang 500 không nằm trong chương 1 (vì `500 > 120`) → **bỏ qua cả chương 1**, còn phải tìm trang thứ `380` từ đầu chương 2. Không nằm trong chương 2 (`380 > 150`) → bỏ qua, còn `230`. Nằm trong chương 3 (`230 <= 300`) → **mở chương 3 ra**, tìm tiếp trong các mục con của nó.

Áp vào cây tiền tố: pre-order thăm **trọn cây con của một nút liền một mạch** (mục 6). Vậy nếu biết **cây con gốc `cur` có bao nhiêu nút** — gọi là `steps` — ta có thể:

- Nếu đáp án **không nằm trong** cây con `cur` → **nhảy qua cả cây con** trong một bước: sang em bên phải `cur + 1`, đồng thời trừ `steps` khỏi số bước còn phải đi.
- Nếu đáp án **nằm trong** cây con `cur` → **đi xuống** con đầu `cur * 10` (tốn đúng 1 bước pre-order).

Mỗi lần nhảy có thể bỏ qua hàng triệu nút. Câu hỏi còn lại: **đếm số nút trong cây con gốc `p` thế nào cho nhanh** (không được duyệt từng nút, vì như vậy lại chậm như cũ)?

## 11. Đếm số nút trong cây con tiền tố `p`: đếm **theo từng tầng**

Nhìn cây con gốc `p = 12` theo từng tầng:

| Tầng | Các số trong tầng | Viết dạng đoạn `[first, last]` |
|---:|---|---|
| 0 | `12` | `[12, 12]` |
| 1 | `120, 121, ..., 129` | `[120, 129]` |
| 2 | `1200, 1201, ..., 1299` | `[1200, 1299]` |
| 3 | `12000, ..., 12999` | `[12000, 12999]` |

**Phát hiện then chốt:** mỗi tầng là một **đoạn số liên tiếp**! Vì "các số có tiền tố `12` và có `2 + t` chữ số" chính là `12` nối với **mọi** đuôi `t` chữ số từ `00..0` tới `99..9`.

Và từ tầng này sang tầng sau, hai biên biến đổi rất đơn giản:

- `first_mới = first * 10` (nối thêm chữ số `0`): `12 → 120 → 1200`.
- `last_mới = last * 10 + 9` (nối thêm chữ số `9`): `12 → 129 → 1299`.

Nhưng ta chỉ đếm các số `<= n`. Với mỗi tầng:

- Nếu `first > n`: tầng này rỗng — và mọi tầng sâu hơn cũng rỗng (vì `first` chỉ tăng) → **dừng**.
- Ngược lại: tầng có `min(last, n) - first + 1` số (cắt biên phải tại `n`).

Ví dụ `n = 1234`, `p = 12`: tầng 0 có `1`; tầng 1 có `129 - 120 + 1 = 10`; tầng 2: `last = 1299` bị cắt về `1234` → `1234 - 1200 + 1 = 35`; tầng 3: `first = 12000 > 1234` → dừng. **Tổng `46`.**

Ví dụ `n = 13`, `p = 1`: tầng 0 `[1, 1]` → 1; tầng 1 `[10, 19]` cắt về `13` → 4; tầng 2 `first = 100 > 13` → dừng. **Tổng `5`** = `{1, 10, 11, 12, 13}` ✓.

In [16]:
def trace_count_steps(n, p):
    first, last, total, level = p, p, 0, 0
    print(f"Đếm cây con gốc p = {p}, n = {n}:")
    while first <= n:
        add = min(last, n) - first + 1
        total += add
        cut = f" (cắt last về n = {n})" if last > n else ""
        print(f"  tầng {level}: [{first}, {last}]{cut} -> +{add}, tổng = {total}")
        first, last, level = first * 10, last * 10 + 9, level + 1
    print(f"  tầng {level}: first = {first} > n -> dừng. Kết quả = {total}\n")
    return total

trace_count_steps(13, 1)
trace_count_steps(1234, 12)
trace_count_steps(100, 1)
trace_count_steps(100, 10)

Đếm cây con gốc p = 1, n = 13:
  tầng 0: [1, 1] -> +1, tổng = 1
  tầng 1: [10, 19] (cắt last về n = 13) -> +4, tổng = 5
  tầng 2: first = 100 > n -> dừng. Kết quả = 5

Đếm cây con gốc p = 12, n = 1234:
  tầng 0: [12, 12] -> +1, tổng = 1
  tầng 1: [120, 129] -> +10, tổng = 11
  tầng 2: [1200, 1299] (cắt last về n = 1234) -> +35, tổng = 46
  tầng 3: first = 12000 > n -> dừng. Kết quả = 46

Đếm cây con gốc p = 1, n = 100:
  tầng 0: [1, 1] -> +1, tổng = 1
  tầng 1: [10, 19] -> +10, tổng = 11
  tầng 2: [100, 199] (cắt last về n = 100) -> +1, tổng = 12
  tầng 3: first = 1000 > n -> dừng. Kết quả = 12

Đếm cây con gốc p = 10, n = 100:
  tầng 0: [10, 10] -> +1, tổng = 1
  tầng 1: [100, 109] (cắt last về n = 100) -> +1, tổng = 2
  tầng 2: first = 1000 > n -> dừng. Kết quả = 2



2

### Hàm `count_steps(n, p)` — viết gọn và kiểm chứng

Tên `count_steps` có nghĩa: *"đi pre-order hết cây con gốc `p` tốn bao nhiêu bước"* = số nút trong cây con đó.

**Độ phức tạp:** mỗi vòng lặp `first` nhân 10, nên số vòng ≤ số chữ số của `n` (`≤ 10` khi `n ≤ 10^9`) → **`O(log n)`** thời gian, `O(1)` bộ nhớ.

Hai ghi chú nhỏ:

- Nếu `p > n` ngay từ đầu, vòng `while` không chạy, kết quả `0` — hợp lý (cây con rỗng). Ta sẽ cần tính chất này.
- Trong C++/Java, `last * 10 + 9` có thể vượt `int` 32-bit khi `n` gần `10^9`, nên phải dùng `long`. Python thì số nguyên không bao giờ tràn.
- Có người viết theo kiểu nửa mở `[first, nxt)` với `nxt = p + 1`, mỗi tầng `nxt *= 10`, cộng `min(nxt, n + 1) - first`. Hoàn toàn tương đương vì `nxt = last + 1`.

In [17]:
def count_steps(n: int, p: int) -> int:
    """Số nút trong cây con tiền tố gốc p (các số trong [1, n] có tiền tố p)."""
    total = 0
    first, last = p, p
    while first <= n:
        total += min(last, n) - first + 1
        first = first * 10
        last = last * 10 + 9
    return total

# Đối chiếu với đếm vét cạn bằng startswith
for n in range(1, 400):
    for p in range(1, n + 3):          # thử cả p > n (phải ra 0)
        assert count_steps(n, p) == subtree_size_brute(n, p), (n, p)
print("count_steps khớp đếm vét cạn với mọi n < 400, mọi p")
print("Kích thước 9 nhánh đầu, n = 100:", [count_steps(100, d) for d in range(1, 10)])
print("count_steps(10**9, 1) =", count_steps(10**9, 1))   # 111111112 = 1 + 10 + ... + 10^8 + 1

count_steps khớp đếm vét cạn với mọi n < 400, mọi p
Kích thước 9 nhánh đầu, n = 100: [12, 11, 11, 11, 11, 11, 11, 11, 11]
count_steps(10**9, 1) = 111111112


## 12. Cách 3 (cách chính) — Nhảy qua cây con hoặc đi xuống

Ghép ý tưởng mục 10 với `count_steps`:

```text
cur = 1          # đang đứng ở số thứ 1 theo thứ tự từ điển
k  -= 1          # k bây giờ = "còn phải đi tới thêm bao nhiêu bước pre-order nữa"
while k > 0:
    steps = count_steps(n, cur)      # số nút trong cây con gốc cur (tính cả cur)
    if steps <= k:                   # đáp án nằm NGOÀI cây con cur
        k   -= steps                 #   đi qua hết cây con: tốn steps bước
        cur += 1                     #   tới em bên phải
    else:                            # đáp án nằm TRONG cây con cur (và không phải chính cur)
        k   -= 1                     #   bước xuống con đầu tốn 1 bước
        cur *= 10
return cur
```

Giải thích từng chi tiết dễ sai:

- **`k -= 1` ở đầu**: ta "đứng sẵn" ở `cur = 1`, tức đã dùng hết 1 trong `k` vị trí. Từ đây `k` mang nghĩa **số bước còn phải đi tiếp**. Khi `k == 0` thì `cur` chính là đáp án.
- **Vì sao `steps <= k` thì nhảy?** Đứng ở `cur`, đi thêm `steps` bước pre-order thì ra khỏi cây con `cur` và tới đúng nút ngay sau nó (em bên phải). Nếu `k >= steps`, đáp án nằm từ chỗ đó trở đi → nhảy. Trường hợp `k == steps` đúng là đáp án là chính `cur + 1` — vẫn phải nhảy, nên dấu là `<=`.
- **Vì sao đi xuống tốn 1 bước?** Theo pre-order, nút ngay sau `cur` là con đầu `cur * 10`. Đi từ `cur` sang `cur * 10` là đúng 1 bước.
- **Vì sao khi `steps > k` thì chắc chắn `cur * 10 <= n`?** Vì `steps > k >= 1` nên `steps >= 2`: cây con `cur` có ít nhất một nút khác `cur`, tức con đầu `cur * 10` tồn tại.

In [18]:
def kth_count_steps(n: int, k: int) -> int:
    cur = 1
    k -= 1
    while k > 0:
        steps = count_steps(n, cur)
        if steps <= k:          # nhảy qua cả cây con cur
            k -= steps
            cur += 1
        else:                   # đi xuống con đầu
            k -= 1
            cur *= 10
    return cur

print(kth_count_steps(13, 2))     # 10 (Ví dụ 1)
print(kth_count_steps(1, 1))      # 1  (Ví dụ 2)
print(kth_count_steps(100, 10))   # 17
print(kth_count_steps(100, 50))   # 53
print(kth_count_steps(10**9, 10**9))   # 999999999 : tức thì!

10
1
17
53
999999999


## 13. Mô phỏng từng bước — `n = 13`, `k = 2` (Ví dụ 1)

Nhắc lại cây (mục 5): nhánh `1` có cây con `{1, 10, 11, 12, 13}` (5 nút), các nhánh `2..9` mỗi nhánh 1 nút.

| Vòng | `cur` trước | `k` trước | `steps = count_steps(13, cur)` | So sánh | Hành động | `cur` sau | `k` sau |
|---:|---:|---:|---|---|---|---:|---:|
| bắt đầu | — | `2` | — | — | đứng ở `1`, `k -= 1` | `1` | `1` |
| 1 | `1` | `1` | `[1,1]→1` + `[10,19]` cắt `13` `→4` = **5** | `5 > 1` | **đi xuống** | `10` | `0` |
| — | | | | `k == 0` | dừng, trả về `cur` | **`10`** | |

Đáp án `10` ✓. Chỉ đúng **1 vòng** — vì `k` nhỏ nên đáp án nằm ngay trong cây con đầu tiên.

Để thấy cả nhánh **nhảy qua**, thử thêm `n = 13, k = 7` (đáp án `3`, thứ tự `1, 10, 11, 12, 13, 2, 3`):

| Vòng | `cur` | `k` trước | `steps` | So sánh | Hành động | `k` sau | `cur` sau |
|---:|---:|---:|---:|---|---|---:|---:|
| bắt đầu | — | 7 | — | — | `k -= 1` | 6 | 1 |
| 1 | `1` | 6 | 5 | `5 <= 6` | **nhảy qua** cây con `1` (bỏ qua `1, 10, 11, 12, 13`) | 1 | `2` |
| 2 | `2` | 1 | 1 | `1 <= 1` | **nhảy qua** cây con `2` (chỉ có `2`) | 0 | `3` |

Đáp án `3` ✓. Hàm `trace_kth` dưới in đúng các dòng này.

In [19]:
def trace_kth(n, k):
    print(f"=== n = {n}, k = {k} ===")
    cur = 1
    k -= 1
    print(f"Bắt đầu: đứng ở cur = 1, còn phải đi k = {k} bước")
    loop = 0
    while k > 0:
        loop += 1
        steps = count_steps(n, cur)
        if steps <= k:
            print(f"Vòng {loop:>2}: cur = {cur:<4} steps = {steps:<4} k = {k:<4} steps <= k -> NHẢY QUA cây con {cur}: k = {k - steps}, cur = {cur + 1}")
            k -= steps
            cur += 1
        else:
            print(f"Vòng {loop:>2}: cur = {cur:<4} steps = {steps:<4} k = {k:<4} steps >  k -> ĐI XUỐNG: k = {k - 1}, cur = {cur * 10}")
            k -= 1
            cur *= 10
    print(f"k = 0 -> đáp án = {cur}\n")
    return cur

trace_kth(13, 2)
trace_kth(13, 7)

=== n = 13, k = 2 ===
Bắt đầu: đứng ở cur = 1, còn phải đi k = 1 bước
Vòng  1: cur = 1    steps = 5    k = 1    steps >  k -> ĐI XUỐNG: k = 0, cur = 10
k = 0 -> đáp án = 10

=== n = 13, k = 7 ===
Bắt đầu: đứng ở cur = 1, còn phải đi k = 6 bước
Vòng  1: cur = 1    steps = 5    k = 6    steps <= k -> NHẢY QUA cây con 1: k = 1, cur = 2
Vòng  2: cur = 2    steps = 1    k = 1    steps <= k -> NHẢY QUA cây con 2: k = 0, cur = 3
k = 0 -> đáp án = 3



3

## 14. Mô phỏng từng bước — `n = 100`, `k = 10`

Thứ tự từ điển của `n = 100` bắt đầu: `1, 10, 100, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, ...`. Đếm tay: số thứ 10 là **`17`**.

Các `count_steps` cần dùng (đã tính ở mục 11): cây con `1` có **12** nút (`1`, `10..19`, `100`); cây con `10` có **2** nút (`10`, `100`); cây con `11..19` mỗi cái **1** nút (vì `110 > 100`).

| Vòng | `cur` | `k` trước | `steps` | So sánh | Hành động | `k` sau | `cur` sau |
|---:|---:|---:|---:|---|---|---:|---:|
| bắt đầu | — | 10 | — | — | `k -= 1` | 9 | `1` |
| 1 | `1` | 9 | 12 | `12 > 9` | **đi xuống** (đáp án ở trong nhánh `1`) | 8 | `10` |
| 2 | `10` | 8 | 2 | `2 <= 8` | **nhảy qua** `{10, 100}` | 6 | `11` |
| 3 | `11` | 6 | 1 | `1 <= 6` | nhảy qua | 5 | `12` |
| 4 | `12` | 5 | 1 | `1 <= 5` | nhảy qua | 4 | `13` |
| 5 | `13` | 4 | 1 | `1 <= 4` | nhảy qua | 3 | `14` |
| 6 | `14` | 3 | 1 | `1 <= 3` | nhảy qua | 2 | `15` |
| 7 | `15` | 2 | 1 | `1 <= 2` | nhảy qua | 1 | `16` |
| 8 | `16` | 1 | 1 | `1 <= 1` | nhảy qua | 0 | `17` |

Đáp án **`17`** ✓. Để ý vòng 2: một lần nhảy bỏ qua **2** nút (`10` và cháu `100`) — đây là chỗ Cách 2 phải đi 2 bước.

Thêm một ví dụ cho thấy sức mạnh nhảy ở **tầng trên cùng**: `n = 100, k = 50` → nhảy qua nguyên nhánh `1` (12 nút), `2`, `3`, `4` (mỗi nhánh 11 nút) chỉ trong 4 vòng, rồi đi xuống nhánh `5` và ra `53`.

In [20]:
trace_kth(100, 10)
trace_kth(100, 50)

=== n = 100, k = 10 ===
Bắt đầu: đứng ở cur = 1, còn phải đi k = 9 bước
Vòng  1: cur = 1    steps = 12   k = 9    steps >  k -> ĐI XUỐNG: k = 8, cur = 10
Vòng  2: cur = 10   steps = 2    k = 8    steps <= k -> NHẢY QUA cây con 10: k = 6, cur = 11
Vòng  3: cur = 11   steps = 1    k = 6    steps <= k -> NHẢY QUA cây con 11: k = 5, cur = 12
Vòng  4: cur = 12   steps = 1    k = 5    steps <= k -> NHẢY QUA cây con 12: k = 4, cur = 13
Vòng  5: cur = 13   steps = 1    k = 4    steps <= k -> NHẢY QUA cây con 13: k = 3, cur = 14
Vòng  6: cur = 14   steps = 1    k = 3    steps <= k -> NHẢY QUA cây con 14: k = 2, cur = 15
Vòng  7: cur = 15   steps = 1    k = 2    steps <= k -> NHẢY QUA cây con 15: k = 1, cur = 16
Vòng  8: cur = 16   steps = 1    k = 1    steps <= k -> NHẢY QUA cây con 16: k = 0, cur = 17
k = 0 -> đáp án = 17

=== n = 100, k = 50 ===
Bắt đầu: đứng ở cur = 1, còn phải đi k = 49 bước
Vòng  1: cur = 1    steps = 12   k = 49   steps <= k -> NHẢY QUA cây con 1: k = 37, cur = 2
Vòng  2:

53

### Nhận xét từ trace — vì sao chỉ cần rất ít vòng?

Mỗi vòng lặp làm **một trong hai** việc:

- **Đi xuống** (`cur *= 10`): `cur` thêm một chữ số. Vì `cur <= n` có tối đa `d` chữ số (`d ≤ 10`), nên đi xuống **tối đa khoảng `d` lần**.
- **Sang phải** (`cur += 1`): ở mỗi tầng, chữ số cuối của `cur` chỉ chạy từ `0` (hoặc `1` ở tầng đầu) đến `9` — và ta không bao giờ nhảy qua em cuối cùng (xem mục 16) → tối đa **9 lần mỗi tầng**.

Tổng số vòng ≤ khoảng `10 · d` ≈ **100** vòng cho `n = 10^9`. Mỗi vòng gọi `count_steps` tốn `O(d)`. Vậy:

- **Time `O(10 · d · d) = O(log² n)`** — cỡ vài trăm đến một nghìn phép tính, bất kể `k`.
- **Space `O(1)`**.

Cell dưới đếm số vòng thực tế trên các input lớn và ngẫu nhiên để kiểm chứng con số "≤ 100".

In [21]:
import random

def count_loops(n, k):
    cur, k, loops = 1, k - 1, 0
    while k > 0:
        loops += 1
        steps = count_steps(n, cur)
        if steps <= k:
            k, cur = k - steps, cur + 1
        else:
            k, cur = k - 1, cur * 10
    return loops

for n, k in [(10**9, 1), (10**9, 10**9), (10**9, 5 * 10**8), (999_999_999, 123_456_789)]:
    print(f"n = {n:>13,}, k = {k:>13,}: {count_loops(n, k):>3} vòng, đáp án = {kth_count_steps(n, k)}")

random.seed(440)
worst = max(count_loops(n, random.randint(1, n)) for n in (random.randint(1, 10**9) for _ in range(3000)))
print("Số vòng lớn nhất trên 3000 input ngẫu nhiên (n <= 10^9):", worst)

n = 1,000,000,000, k =             1:   0 vòng, đáp án = 1
n = 1,000,000,000, k = 1,000,000,000:  88 vòng, đáp án = 999999999
n = 1,000,000,000, k =   500,000,000:  78 vòng, đáp án = 549999998
n =   999,999,999, k =   123,456,789:  18 vòng, đáp án = 211111103
Số vòng lớn nhất trên 3000 input ngẫu nhiên (n <= 10^9): 73


## 15. Code nộp LeetCode

Chuyển `kth_count_steps` vào đúng khung `class Solution`. `count_steps` viết thành hàm lồng bên trong method (hoặc một method riêng `self._count_steps` cũng được) để bản nộp tự đủ, không phụ thuộc cell nào khác.

Các test gồm: hai ví dụ của đề; biên `k = 1` (luôn ra `1`), `k = n` (luôn ra số **lớn nhất về từ điển**); `n = 10` (số `10` chen giữa `1` và `2`); và các ca `n = 10^9`.

In [22]:
class Solution:
    def findKthNumber(self, n: int, k: int) -> int:
        def count_steps(p: int) -> int:
            total, first, last = 0, p, p
            while first <= n:
                total += min(last, n) - first + 1
                first, last = first * 10, last * 10 + 9
            return total

        cur = 1
        k -= 1
        while k > 0:
            steps = count_steps(cur)
            if steps <= k:
                k -= steps
                cur += 1
            else:
                k -= 1
                cur *= 10
        return cur


sol = Solution()
assert sol.findKthNumber(13, 2) == 10          # Ví dụ 1
assert sol.findKthNumber(1, 1) == 1            # Ví dụ 2
assert sol.findKthNumber(13, 13) == 9          # k = n: số cuối theo từ điển
assert sol.findKthNumber(10, 2) == 10          # 1, 10, 2, 3, ...
assert sol.findKthNumber(10, 3) == 2
assert sol.findKthNumber(100, 10) == 17
assert sol.findKthNumber(100, 3) == 100
assert sol.findKthNumber(10**9, 1) == 1
assert sol.findKthNumber(10**9, 2) == 10
assert sol.findKthNumber(10**9, 10) == 1_000_000_000  # 1, 10, 100, ..., 10^9
assert sol.findKthNumber(10**9, 11) == 100_000_001    # sau 10^9 phải lùi lên 10^8 rồi +1
assert sol.findKthNumber(10**9, 10**9) == 999_999_999
print("Solution: tất cả test PASS")

Solution: tất cả test PASS


### Bản tham khảo — Cách 2b dạng `class` (KHÔNG nên nộp)

Để so sánh, đây là Cách 2b viết theo khung LeetCode (khi nộp phải đổi tên lớp thành `Solution`). Nó **đúng** nhưng sẽ **TLE** với các test có `k` cỡ `10^8`–`10^9`. Giữ lại vì đây là lời giải chuẩn của bài #386 (chỉ cần gom các `cur` vào list).

In [23]:
class SolutionWalk:
    def findKthNumber(self, n: int, k: int) -> int:
        cur = 1
        for _ in range(k - 1):
            if cur * 10 <= n:
                cur *= 10
            else:
                while cur % 10 == 9 or cur + 1 > n:
                    cur //= 10
                cur += 1
        return cur

w = SolutionWalk()
assert w.findKthNumber(13, 2) == 10 and w.findKthNumber(1, 1) == 1
assert w.findKthNumber(100, 50) == 53 and w.findKthNumber(10**9, 1000) == kth_count_steps(10**9, 1000)
print("SolutionWalk: PASS (nhưng O(k) -> TLE khi k lớn)")

SolutionWalk: PASS (nhưng O(k) -> TLE khi k lớn)


### Đối chiếu chéo các cách (test chéo)

Ba tầng kiểm tra, có `random.seed` cố định để chạy lại ra đúng kết quả:

1. **Toàn bộ** cặp `(n, k)` với `n ≤ 300`: so Cách 3 (và `Solution`) với trọng tài Cách 1 (sort).
2. `n` vừa (tới `2 · 10^5`), `k` ngẫu nhiên: so Cách 3 với Cách 2a (DFS) và 2b (walk) — sort ở cỡ này vẫn chạy được nhưng 2a/2b nhanh hơn khi `k` không quá lớn.
3. **Tính chất hoán vị**: với một `n`, gọi Cách 3 cho mọi `k = 1..n` phải ra đúng một hoán vị của `1..n` và trùng thứ tự từ điển.

In [24]:
# 1. Mọi (n, k) với n <= 300 so với trọng tài sort
for n in range(1, 301):
    order = lex_order_sort(n)
    for k in range(1, n + 1):
        assert kth_count_steps(n, k) == order[k - 1] == sol.findKthNumber(n, k), (n, k)
print("Tầng 1: Cách 3 == Cách 1 với mọi n <= 300, mọi k")

# 2. n vừa, k ngẫu nhiên: so với Cách 2a và 2b
random.seed(2024)
for _ in range(150):
    n = random.randint(1, 200_000)
    k = random.randint(1, min(n, 20_000))
    expected = kth_walk(n, k)
    assert kth_count_steps(n, k) == expected == kth_dfs(n, k), (n, k)
print("Tầng 2: Cách 3 == Cách 2a == Cách 2b trên 150 input ngẫu nhiên")

# 3. Tính chất hoán vị
n = 5000
seq = [kth_count_steps(n, k) for k in range(1, n + 1)]
assert sorted(seq) == list(range(1, n + 1)) and seq == lex_order_iter(n)
print("Tầng 3: k = 1..5000 cho đúng thứ tự từ điển của 1..5000")

Tầng 1: Cách 3 == Cách 1 với mọi n <= 300, mọi k


Tầng 2: Cách 3 == Cách 2a == Cách 2b trên 150 input ngẫu nhiên
Tầng 3: k = 1..5000 cho đúng thứ tự từ điển của 1..5000


## 16. Vì sao Cách 3 đúng?

**Invariant (bất biến)** — luôn đúng ở đầu mỗi vòng `while`:

> *Đáp án là nút đứng **sau `cur` đúng `k` bước** theo thứ tự pre-order. Hơn nữa, đáp án nằm trong cây con của **cha** của `cur` (với `cur` ở tầng đầu thì "cha" là gốc ảo), ở vị trí từ `cur` trở về sau.*

**Lúc bắt đầu:** `cur = 1` là số thứ 1, `k` đã trừ 1 → đáp án ở sau `cur` đúng `k` bước. ✓ (Và vì `k ≤ n` nên đáp án tồn tại trong toàn cây.)

**Nhánh nhảy (`steps <= k`):** pre-order thăm trọn cây con `cur` (`steps` nút, bắt đầu từ chính `cur`) rồi mới tới nút kế tiếp. Đi `steps` bước từ `cur` thì vừa ra khỏi cây con và tới **nút kế tiếp sau cây con**. Nút đó có phải là `cur + 1` không? Có, *miễn là `cur` chưa phải em cuối cùng*. Vì đáp án nằm trong cây con của cha (invariant) mà không nằm trong cây con `cur` và các anh trước nó, nên vẫn còn em phía sau chứa đáp án → `cur + 1` tồn tại và đúng là nút kế tiếp. Gán `k -= steps`, `cur += 1` → invariant giữ nguyên. ✓

**Nhánh đi xuống (`steps > k`):** đáp án nằm trong cây con `cur` nhưng không phải `cur` (vì `k ≥ 1`) → nằm trong cây con của một con nào đó. Nút ngay sau `cur` là con đầu `cur * 10` (tồn tại vì `steps ≥ 2`). Gán `k -= 1`, `cur *= 10` → đáp án vẫn sau `cur` đúng `k` bước và nằm trong cây con của cha mới (`cur` cũ). ✓

**Kết thúc:** mỗi vòng `k` giảm ít nhất 1 nên vòng lặp dừng; khi `k == 0` thì "đáp án đứng sau `cur` 0 bước" tức **chính là `cur`**. ✓

Hệ quả của invariant: ta **không bao giờ** nhảy từ em cuối (ví dụ từ `19` sang `20` — sai cha), và **không bao giờ** đứng ở một `cur > n`. Lý do: ngay lúc đi xuống cây con của một nút, `k` nhỏ hơn tổng kích thước các cây con con của nó, nên ta không thể "tiêu" hết các con rồi rơi ra ngoài.

## 17. So sánh các cách

Ký hiệu: `n` là cận trên của đoạn, `k` là thứ tự cần tìm, `d = số chữ số của n ≈ log₁₀ n` (`d ≤ 10`).

| Cách | Ý tưởng | Time | Space phụ | `n = k = 10^9` | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Sort `key=str` | Dịch thẳng định nghĩa | `O(n log n · d)` | `O(n)` | MLE / TLE | Trọng tài khi test, `n` nhỏ |
| 2a. DFS đệ quy đếm tới `k` | Pre-order = thứ tự từ điển | `O(k)` | `O(d)` stack | TLE | `k` nhỏ; nền tảng cho #386 |
| 2b. Vòng lặp ×10 / +1 / ÷10 | Tính "số kế tiếp" trực tiếp | `O(k)` | `O(1)` | TLE | Lời giải chuẩn của #386 |
| **3. `count_steps` + nhảy/đi xuống** | Đếm cây con theo tầng, bỏ qua cả cây con | **`O(d²) = O(log² n)`** | **`O(1)`** | **~100 vòng, tức thì** | **Nộp bài 440** |

Bài học về tư duy: từ Cách 1 → 2 ta bỏ được việc **xử lý những phần không cần** (chỉ cần `k` số đầu). Từ Cách 2 → 3 ta bỏ được việc **đi qua từng phần tử một**: thay bằng **đếm cả khối** rồi nhảy. "Đếm thay vì liệt kê" là kỹ thuật cốt lõi của rất nhiều bài "tìm phần tử thứ `k`".

## 18. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `k -= 1` ở đầu (lệch một vị trí)

Bắt đầu với `cur = 1` nghĩa là ta **đã đứng ở số thứ 1**. Nếu không trừ 1, `k` vẫn mang nghĩa "số thứ tự" thay vì "số bước còn phải đi" → mọi đáp án bị dịch sau **một vị trí**.

Với `n = 13, k = 2`: bản sai đi xuống `10` (k còn 1), rồi nhảy qua `10` (k còn 0) → trả về `11` thay vì `10`.

In [25]:
def wrong_no_initial_decrement(n, k):
    cur = 1
    # THIẾU: k -= 1
    while k > 0:
        steps = count_steps(n, cur)
        if steps <= k:
            k, cur = k - steps, cur + 1
        else:
            k, cur = k - 1, cur * 10
    return cur

for n, k in [(13, 2), (13, 1), (100, 10)]:
    print(f"n = {n}, k = {k}: bản sai -> {wrong_no_initial_decrement(n, k)}, đúng -> {kth_count_steps(n, k)}")

n = 13, k = 2: bản sai -> 11, đúng -> 10
n = 13, k = 1: bản sai -> 10, đúng -> 1
n = 100, k = 10: bản sai -> 18, đúng -> 17


### Lỗi 2 — Dùng `steps < k` thay vì `steps <= k`

Khi `steps == k`, đáp án chính là **nút ngay sau cây con `cur`**, tức `cur + 1` → phải **nhảy**. Nếu viết `<`, ta lại đi xuống cây con `cur` và cuối cùng có thể "rơi" ra một số **lớn hơn `n`**!

Với `n = 13, k = 6` (đáp án `2`): sau `k -= 1` thì `k = 5`, `steps(1) = 5`. Bản sai thấy `5 < 5` sai nên đi xuống `10` (`k = 4`), rồi nhảy sang `11`, `12`, `13` (`k` còn `1`); tới `13` thì `steps = 1 < 1` sai, lại "đi xuống" `130` — **một số không hề nằm trong `[1, 13]`**.

In [26]:
def wrong_strict_less(n, k):
    cur = 1
    k -= 1
    while k > 0:
        steps = count_steps(n, cur)
        if steps < k:                       # SAI: phải là <=
            k, cur = k - steps, cur + 1
        else:
            k, cur = k - 1, cur * 10
    return cur

for n, k in [(13, 6), (13, 7), (100, 13)]:
    print(f"n = {n}, k = {k}: bản sai -> {wrong_strict_less(n, k)}, đúng -> {kth_count_steps(n, k)}")

n = 13, k = 6: bản sai -> 130, đúng -> 2
n = 13, k = 7: bản sai -> 20, đúng -> 3
n = 100, k = 13: bản sai -> 190, đúng -> 2


### Lỗi 3 — `count_steps` quên cắt `min(last, n)` hoặc cập nhật biên sai

Ba phiên bản sai hay gặp:

- **(a)** Cộng `last - first + 1` mà không cắt tại `n` → đếm cả những số `> n`. Với `n = 13, p = 1`: tầng 1 đếm đủ `10..19` = 10 số → tổng `11` thay vì `5`.
- **(b)** Cập nhật `last = last * 10` (quên `+ 9`) → tầng 1 chỉ còn `[10, 10]` → thiếu số.
- **(c)** Điều kiện vòng lặp `while last <= n` thay vì `while first <= n` → bỏ mất tầng bị cắt dở (tầng có `first <= n < last`).

In [27]:
def steps_no_min(n, p):          # (a)
    total, first, last = 0, p, p
    while first <= n:
        total += last - first + 1
        first, last = first * 10, last * 10 + 9
    return total

def steps_no_plus9(n, p):        # (b)
    total, first, last = 0, p, p
    while first <= n:
        total += min(last, n) - first + 1
        first, last = first * 10, last * 10
    return total

def steps_wrong_cond(n, p):      # (c)
    total, first, last = 0, p, p
    while last <= n:
        total += min(last, n) - first + 1
        first, last = first * 10, last * 10 + 9
    return total

for n, p in [(13, 1), (100, 1), (1234, 12)]:
    print(f"n = {n:>4}, p = {p:>2}: đúng = {count_steps(n, p):>3} | (a) = {steps_no_min(n, p):>4} "
          f"| (b) = {steps_no_plus9(n, p):>3} | (c) = {steps_wrong_cond(n, p):>3}")

n =   13, p =  1: đúng =   5 | (a) =   11 | (b) =   2 | (c) =   1
n =  100, p =  1: đúng =  12 | (a) =  111 | (b) =   3 | (c) =  11
n = 1234, p = 12: đúng =  46 | (a) =  111 | (b) =   3 | (c) =  11


### Lỗi 4 — Đi xuống mà quên `k -= 1`

Đi từ `cur` xuống `cur * 10` là **một bước** pre-order (đã thăm thêm một nút). Quên trừ thì `k` "còn dư" một bước → đáp án lệch sau một vị trí. Với `n = 13, k = 2`: bản sai xuống `10` mà `k` vẫn `1`, rồi nhảy qua `10` → `11`.

### Lỗi 5 — `next_lex` (Cách 2b) quên điều kiện `cur + 1 > n`

Chỉ kiểm tra `cur % 10 == 9` là chưa đủ: ở `n = 13`, `cur = 13`, bản sai sang thẳng `14` — vượt `n`. Danh sách sinh ra sẽ chứa các số "ma" `14, 15, ...`.

In [28]:
def wrong_down_no_decrement(n, k):
    cur = 1
    k -= 1
    while k > 0:
        steps = count_steps(n, cur)
        if steps <= k:
            k, cur = k - steps, cur + 1
        else:
            cur *= 10                      # SAI: quên k -= 1
    return cur

def wrong_next_lex(cur, n):
    if cur * 10 <= n:
        return cur * 10
    while cur % 10 == 9:                   # SAI: thiếu "or cur + 1 > n"
        cur //= 10
    return cur + 1

print("Lỗi 4:", [(k, wrong_down_no_decrement(13, k), kth_count_steps(13, k)) for k in (2, 3)])
seq, cur = [1], 1
for _ in range(12):
    cur = wrong_next_lex(cur, 13)
    seq.append(cur)
print("Lỗi 5, n = 13:", seq)
print("Đúng  , n = 13:", lex_order_iter(13))

Lỗi 4: [(2, 11, 10), (3, 12, 11)]
Lỗi 5, n = 13: [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 3]
Đúng  , n = 13: [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]


### Các lỗi khác cần nhớ

- **Cách 1 trả `order[k]`** thay vì `order[k - 1]` → lệch một (và `IndexError` khi `k = n`).
- **Cách 1 sort theo số** (`sorted(range(1, n + 1))`, thiếu `key=str`) → ra thứ tự tăng dần thông thường: `n = 13, k = 2` trả `2` thay vì `10`.
- **Bắt đầu từ `cur = 0`** hoặc coi `0` là một nút → sai, vì đoạn là `[1, n]`, và `0 * 10 = 0` khiến vòng lặp kẹt.
- **Đếm cây con bằng cách duyệt từng nút** (DFS đếm) bên trong vòng lặp → mất hết lợi thế, lại thành `O(n)`.
- **Viết C++/Java** mà để `first`, `last` là `int` → tràn số khi `last * 10 + 9` vượt `2^31 - 1`; phải dùng `long`.

## 19. Mẹo nhận dạng pattern

- **Dấu hiệu:** đề hỏi "phần tử **thứ `k`**" trong một thứ tự rất dài (tới `10^9` phần tử) mà **không thể liệt kê**. → Nghĩ ngay tới **"đếm cả khối rồi nhảy"**: chia không gian thành các khối có thể **đếm nhanh kích thước**, bỏ qua cả khối khi `k` lớn hơn kích thước, đi vào khối khi `k` rơi vào trong.
- **"Thứ tự từ điển" + số nguyên** → hình dung **cây tiền tố 10 nhánh**; thứ tự từ điển = **pre-order**; cây con của `p` = các số có tiền tố `p` = mỗi tầng là **một đoạn liên tiếp `[p·10^t, p·10^t + 10^t − 1]`**.
- **Đếm số phần tử trong một khối** thường quy về đếm các đoạn liên tiếp và **cắt biên tại `n`** (`min(last, n)`).
- Luôn tự hỏi: **`k` đang mang nghĩa "thứ tự" (1-based) hay "số bước còn lại" (0-based)?** — đây là gốc của mọi lỗi off-by-one ở bài loại này.

## 20. Biến thể và bài liên quan

| Bài | Độ khó | Liên hệ với bài này |
|---|---|---|
| LeetCode 386 — Lexicographical Numbers | Medium | Liệt kê **hết** `1..n` theo từ điển — chính là Cách 2 (DFS hoặc vòng lặp ×10/+1/÷10), `O(n)` thời gian, `O(1)` bộ nhớ phụ |
| Bài ngược: "số `x` đứng thứ mấy theo từ điển?" | — | Cộng `count_steps` của các anh em đứng trước trên đường từ gốc tới `x` — Bài tập 5 |
| Thứ tự từ điển theo **cơ số `b`** | — | Thay `10` bằng `b`, `+9` bằng `+(b−1)` — Bài tập 6 |
| LeetCode 60 — Permutation Sequence | Hard | Cũng "tìm phần tử thứ `k`" bằng cách đếm kích thước khối (`(n−1)!` hoán vị mỗi khối) rồi nhảy |
| LeetCode 1643 — Kth Smallest Instructions | Hard | Tìm chuỗi đường đi thứ `k` theo từ điển bằng cách đếm tổ hợp `C(a+b, a)` cho mỗi lựa chọn |
| LeetCode 1415 — The k-th Lexicographical String of All Happy Strings of Length n | Medium | Đếm số chuỗi trong mỗi nhánh (`2^(n−1)`...) rồi chọn nhánh chứa `k` |
| LeetCode 668 — Kth Smallest Number in Multiplication Table | Hard | Một kiểu "tìm thứ `k`" khác: binary search trên đáp án + hàm đếm |
| LeetCode 208 / 212 — Trie | Medium / Hard | Cây tiền tố tổng quát cho chuỗi; ở bài 440 cây tiền tố là **ẩn** (không cần xây), chỉ dùng công thức |

## 21. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 22) và **Đáp án** (mục 23, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

```python
print("10" < "9", 10 < 9)                      # (a)
print(sorted([3, 21, 100, 9], key=str))        # (b)
print(sorted(["b", "ab", "a", "abc"]))         # (c)
print("123".startswith("12"), "12".startswith("123"))   # (d)
```

(e) Với `n = 20`: số đứng ngay sau `19` theo thứ tự từ điển là số nào? Ngay sau `2`? Ngay sau `20`?

### Bài 2 — Tính tay

(a) Dùng bảng tầng `[first, last]` như mục 11, tính `count_steps(234, 2)` và `count_steps(234, 3)`.

(b) Lập bảng trace như mục 13–14 (cột `cur`, `k` trước, `steps`, so sánh, hành động, `k` sau, `cur` sau) cho **`n = 25, k = 13`** và tìm đáp án.

Chỉ dùng giấy bút; phần đáp án có code kiểm tra.

### Bài 3 — Điền TODO cho cách chính

Hoàn thiện `exercise_count_steps(n, p)` và `exercise_find_kth(n, k)` theo Cách 3, không nhìn lại mục 11–12.

In [29]:
def exercise_count_steps(n: int, p: int) -> int:
    total = 0
    first, last = p, p
    # TODO 1: while ...:                 (khi nào tầng còn có số <= n?)
    #     TODO 2: total += ...            (nhớ cắt biên phải tại n)
    #     TODO 3: cập nhật first, last sang tầng kế tiếp
    raise NotImplementedError("Hãy tự hoàn thành Bài 3 (count_steps)")

def exercise_find_kth(n: int, k: int) -> int:
    cur = 1
    # TODO 4: đổi nghĩa k thành "số bước còn phải đi"
    # TODO 5: while k > 0: tính steps, rồi NHẢY QUA (k -= steps, cur += 1) hoặc ĐI XUỐNG (k -= 1, cur *= 10)
    raise NotImplementedError("Hãy tự hoàn thành Bài 3 (find_kth)")

# Tự làm xong rồi mới mở test:
# assert exercise_count_steps(13, 1) == 5
# assert exercise_count_steps(1234, 12) == 46
# assert exercise_count_steps(13, 14) == 0
# assert exercise_find_kth(13, 2) == 10
# assert exercise_find_kth(1, 1) == 1
# assert exercise_find_kth(100, 10) == 17
# assert exercise_find_kth(10**9, 10**9) == 999999999

### Bài 4 — Tự viết lại #386 Lexicographical Numbers (không sort, không đệ quy)

Viết `exercise_lexical_order(n)` trả về list `1..n` theo thứ tự từ điển, dùng **vòng lặp** với 3 luật "nhân 10 / cộng 1 / chia 10" (mục 9). Không dùng `sorted`, không dùng đệ quy.

```python
lexical_order(13) -> [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
lexical_order(2)  -> [1, 2]
```

In [30]:
def exercise_lexical_order(n: int) -> list:
    # TODO: bắt đầu cur = 1; lặp n lần: thêm cur vào kết quả, rồi tính số kế tiếp
    pass

# Gợi ý test:
# assert exercise_lexical_order(13) == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
# assert exercise_lexical_order(2) == [1, 2]
# for n in range(1, 500):
#     assert exercise_lexical_order(n) == sorted(range(1, n + 1), key=str)

### Bài 5 — Bài ngược: số `x` đứng thứ mấy?

Viết `exercise_rank(n, x)` (với `1 ≤ x ≤ n ≤ 10^9`) trả về vị trí (đếm từ 1) của `x` trong thứ tự từ điển của `1..n`. Phải chạy nhanh với `n = 10^9` (không được liệt kê).

```python
rank(13, 10) -> 2
rank(13, 2)  -> 6        # 1, 10, 11, 12, 13, 2
rank(100, 17) -> 10
rank(100, 53) -> 50
rank(10**9, 999999999) -> 1000000000
```

Tính chất kiểm tra: `rank(n, findKthNumber(n, k)) == k` với mọi `k`.

In [31]:
def exercise_rank(n: int, x: int) -> int:
    # TODO: đi từ gốc xuống x theo các tiền tố của str(x);
    #       ở mỗi tầng, cộng kích thước cây con của các anh em đứng TRƯỚC tiền tố hiện tại, rồi cộng 1 cho chính tiền tố
    pass

# Gợi ý test:
# assert exercise_rank(13, 10) == 2
# assert exercise_rank(13, 2) == 6
# assert exercise_rank(100, 17) == 10
# assert exercise_rank(100, 53) == 50
# assert exercise_rank(10**9, 999999999) == 10**9

### Bài 6 — Mini challenge: thứ tự từ điển theo cơ số `b`

Viết mỗi số `1..n` trong **cơ số `b`** (`2 ≤ b ≤ 10`), sắp xếp theo thứ tự từ điển của các chuỗi đó. Viết `exercise_find_kth_base(n, k, b)` trả về số (giá trị thập phân) đứng thứ `k`. Phải chạy nhanh với `n = 10^9`.

Ví dụ `b = 2, n = 5`: các chuỗi nhị phân `1, 10, 11, 100, 101` → thứ tự từ điển `"1" < "10" < "100" < "101" < "11"` → các số `1, 2, 4, 5, 3`.

```python
find_kth_base(5, 3, 2)    -> 4
find_kth_base(5, 5, 2)    -> 3
find_kth_base(13, 2, 10)  -> 10     # b = 10 chính là bài gốc
```

In [32]:
def exercise_find_kth_base(n: int, k: int, b: int) -> int:
    # TODO: tổng quát count_steps và vòng lặp chính cho cơ số b
    pass

# Gợi ý test:
# assert exercise_find_kth_base(5, 3, 2) == 4
# assert exercise_find_kth_base(5, 5, 2) == 3
# assert exercise_find_kth_base(13, 2, 10) == 10

## 22. Gợi ý theo mức

**Bài 1**
1. Xem lại bảng so sánh chuỗi ở mục 2: so ký tự đầu trước.
2. (b) so các chuỗi `"3", "21", "100", "9"`; (c) chuỗi là tiền tố đứng trước chuỗi dài hơn.
3. (e) dùng 3 luật của `next_lex`: `190 > 20` và `19` tận cùng 9 → lùi; `2 * 10 = 20 <= 20` → xuống; `20` không xuống được, `21 > 20` → lùi.

**Bài 2**
1. (a) Mỗi tầng: `first *= 10`, `last = last * 10 + 9`; dừng khi `first > 234`.
2. (b) Cây con `1` với `n = 25` gồm `1, 10..19` → 11 nút; cây con `2` gồm `2, 20..25` → 7 nút.
3. (b) Sau `k -= 1` thì `k = 12`; `11 <= 12` → nhảy; rồi `7 > 1` → đi xuống.

**Bài 3**
1. Tầng còn số khi `first <= n`; số lượng là `min(last, n) - first + 1`.
2. `first, last = first * 10, last * 10 + 9`.
3. `k -= 1` trước vòng lặp; so sánh `steps <= k`.

**Bài 4**
1. Lặp đúng `n` lần; mỗi lần `append(cur)` trước rồi mới tính số kế tiếp.
2. Luật 1: `if cur * 10 <= n: cur *= 10`.
3. Luật 2–3: `while cur % 10 == 9 or cur + 1 > n: cur //= 10` rồi `cur += 1`. (Lần lặp cuối cùng `cur` sẽ "quay vòng" về giá trị vô nghĩa — không sao vì không `append` nữa.)

**Bài 5**
1. Các số đứng trước `x` (tính cả `x`) gồm: các **tổ tiên** của `x` (tiền tố của `str(x)`, tính cả `x`) và **toàn bộ cây con** của mọi nút "anh" nằm bên trái đường đi từ gốc tới `x`.
2. Với tiền tố độ dài `L` là `q` (cha là `q // 10`): các anh của `q` là `start, ..., q − 1`, với `start = 1` nếu `L = 1`, ngược lại `start = (q // 10) * 10`.
3. `rank += sum(count_steps(n, s) for s in range(start, q)) + 1` cho từng `L`.

**Bài 6**
1. Cây tiền tố cơ số `b`: con của `x` là `b·x, b·x + 1, ..., b·x + (b − 1)`.
2. Trong `count_steps`: `first *= b`, `last = last * b + (b − 1)`.
3. Vòng lặp chính giữ nguyên, chỉ đổi `cur *= 10` thành `cur *= b`. Để kiểm tra, viết hàm đổi sang cơ số `b` bằng `% b` và `// b` rồi sort theo chuỗi đó.

## 23. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

- (a) `True False` — chuỗi: `'1' < '9'`; số: `10 > 9`.
- (b) `[100, 21, 3, 9]` — so `"100" < "21" < "3" < "9"`; kết quả vẫn là số.
- (c) `['a', 'ab', 'abc', 'b']` — `"a"` là tiền tố của `"ab"`, `"ab"` là tiền tố của `"abc"`; cả ba bắt đầu bằng `'a' < 'b'`.
- (d) `True False`.
- (e) Với `n = 20`: sau `19` là **`2`** (lùi lên `1` rồi `+1`); sau `2` là **`20`** (`20 <= 20`, đi xuống); sau `20` là **`3`** (`200 > 20`, `21 > 20` → lùi về `2` → `+1`).

### Đáp án Bài 2

(a) `count_steps(234, 2)`: tầng 0 `[2, 2]` → 1; tầng 1 `[20, 29]` → 10; tầng 2 `[200, 299]` cắt về `234` → `234 − 200 + 1 = 35`; tầng 3 `first = 2000 > 234` dừng → **46**.
`count_steps(234, 3)`: `[3, 3]` → 1; `[30, 39]` → 10; tầng 2 `first = 300 > 234` dừng → **11**.

(b) `n = 25, k = 13` (thứ tự: `1, 10, ..., 19, 2, 20, ...` → số thứ 13 là `20`):

| Vòng | `cur` | `k` trước | `steps` | So sánh | Hành động | `k` sau | `cur` sau |
|---:|---:|---:|---:|---|---|---:|---:|
| bắt đầu | — | 13 | — | — | `k -= 1` | 12 | `1` |
| 1 | `1` | 12 | 11 | `11 <= 12` | nhảy qua cây con `1` | 1 | `2` |
| 2 | `2` | 1 | 7 | `7 > 1` | đi xuống | 0 | `20` |

Đáp án **`20`**.

### Đáp án Bài 3 → Bài 6 (code bên dưới, có assert)

In [33]:
# ---------- Kiểm tra lại Bài 1 và Bài 2 bằng code ----------
assert ("10" < "9", 10 < 9) == (True, False)
assert sorted([3, 21, 100, 9], key=str) == [100, 21, 3, 9]
assert sorted(["b", "ab", "a", "abc"]) == ["a", "ab", "abc", "b"]
assert ("123".startswith("12"), "12".startswith("123")) == (True, False)
assert (next_lex(19, 20), next_lex(2, 20), next_lex(20, 20)) == (2, 20, 3)
assert count_steps(234, 2) == 46 and count_steps(234, 3) == 11
assert kth_count_steps(25, 13) == 20 == kth_sort(25, 13)

# ---------- Bài 3 ----------
def answer_count_steps(n: int, p: int) -> int:
    total = 0
    first, last = p, p
    while first <= n:
        total += min(last, n) - first + 1
        first, last = first * 10, last * 10 + 9
    return total

def answer_find_kth(n: int, k: int) -> int:
    cur = 1
    k -= 1
    while k > 0:
        steps = answer_count_steps(n, cur)
        if steps <= k:
            k -= steps
            cur += 1
        else:
            k -= 1
            cur *= 10
    return cur

assert answer_count_steps(13, 1) == 5
assert answer_count_steps(1234, 12) == 46
assert answer_count_steps(13, 14) == 0
assert answer_find_kth(13, 2) == 10
assert answer_find_kth(1, 1) == 1
assert answer_find_kth(100, 10) == 17
assert answer_find_kth(10**9, 10**9) == 999999999

# ---------- Bài 4 ----------
def answer_lexical_order(n: int) -> list:
    result = []
    cur = 1
    for _ in range(n):
        result.append(cur)
        if cur * 10 <= n:
            cur *= 10
        else:
            while cur % 10 == 9 or cur + 1 > n:
                cur //= 10
            cur += 1
    return result

assert answer_lexical_order(13) == [1, 10, 11, 12, 13, 2, 3, 4, 5, 6, 7, 8, 9]
assert answer_lexical_order(2) == [1, 2]
for n in range(1, 500):
    assert answer_lexical_order(n) == sorted(range(1, n + 1), key=str)

print("Bài 1 -> Bài 4: PASS")

Bài 1 -> Bài 4: PASS


In [34]:
# ---------- Bài 5 ----------
def answer_rank(n: int, x: int) -> int:
    s = str(x)
    rank = 0
    for L in range(1, len(s) + 1):
        q = int(s[:L])                          # tiền tố độ dài L (tổ tiên của x, hoặc chính x)
        start = 1 if L == 1 else (q // 10) * 10 # anh cả của q
        for sib in range(start, q):             # các anh đứng trước q: cả cây con đều đứng trước x
            rank += count_steps(n, sib)
        rank += 1                               # chính q đứng trước (hoặc là) x
    return rank

assert answer_rank(13, 10) == 2
assert answer_rank(13, 2) == 6
assert answer_rank(100, 17) == 10
assert answer_rank(100, 53) == 50
assert answer_rank(10**9, 999999999) == 10**9
for n in range(1, 300):                         # rank là hàm ngược của find_kth
    for k in range(1, n + 1):
        assert answer_rank(n, kth_count_steps(n, k)) == k, (n, k)
random.seed(7)
for _ in range(300):
    n = random.randint(1, 10**9)
    k = random.randint(1, n)
    assert answer_rank(n, kth_count_steps(n, k)) == k

# ---------- Bài 6 ----------
def to_base(x: int, b: int) -> str:
    digits = []
    while x > 0:
        digits.append(str(x % b))
        x //= b
    return "".join(reversed(digits))

def answer_find_kth_base(n: int, k: int, b: int) -> int:
    def steps_base(p):
        total, first, last = 0, p, p
        while first <= n:
            total += min(last, n) - first + 1
            first, last = first * b, last * b + (b - 1)
        return total

    cur = 1
    k -= 1
    while k > 0:
        steps = steps_base(cur)
        if steps <= k:
            k, cur = k - steps, cur + 1
        else:
            k, cur = k - 1, cur * b
    return cur

assert answer_find_kth_base(5, 3, 2) == 4
assert answer_find_kth_base(5, 5, 2) == 3
assert answer_find_kth_base(13, 2, 10) == 10
for b in range(2, 11):                          # đối chiếu với sort theo chuỗi cơ số b
    for n in range(1, 150):
        order = sorted(range(1, n + 1), key=lambda x: to_base(x, b))
        for k in range(1, n + 1):
            assert answer_find_kth_base(n, k, b) == order[k - 1], (n, k, b)
print("Bài 5 -> Bài 6: PASS")
print("Ví dụ: trong 1..10^9, số 2026 đứng thứ", f"{answer_rank(10**9, 2026):,}", "theo từ điển")
print("Ví dụ: cơ số 2, n = 10^9, k = 10^9 ->", answer_find_kth_base(10**9, 10**9, 2))

Bài 5 -> Bài 6: PASS
Ví dụ: trong 1..10^9, số 2026 đứng thứ 114,000,004 theo từ điển
Ví dụ: cơ số 2, n = 10^9, k = 10^9 -> 536870911


## 24. Template ngắn dễ nhớ

```python
class Solution:
    def findKthNumber(self, n: int, k: int) -> int:
        def count_steps(p):                      # số nút cây con tiền tố p
            total, first, last = 0, p, p
            while first <= n:                    # tầng còn số <= n
                total += min(last, n) - first + 1  # cắt biên phải tại n
                first, last = first * 10, last * 10 + 9
            return total

        cur, k = 1, k - 1                        # đứng ở số thứ 1; k = số bước còn lại
        while k > 0:
            steps = count_steps(cur)
            if steps <= k:                       # đáp án ngoài cây con cur
                k, cur = k - steps, cur + 1      #   -> nhảy qua cả cây con
            else:                                # đáp án trong cây con cur
                k, cur = k - 1, cur * 10         #   -> đi xuống con đầu
        return cur
```

Câu thần chú: **"Thứ tự từ điển = pre-order cây 10 nhánh. Đếm cây con theo tầng `[first, last]` (×10, ×10+9, cắt tại `n`). Đủ bước thì nhảy sang phải (`+1`), không đủ thì chui xuống (`×10`, trừ 1)."**

## 25. Checklist trước khi submit

- [ ] Đã `k -= 1` trước vòng lặp (vì đang đứng sẵn ở `cur = 1`)?
- [ ] So sánh là `steps <= k` (có dấu `=`)?
- [ ] Đi xuống có `k -= 1`, nhảy qua có `k -= steps`?
- [ ] `count_steps`: điều kiện `while first <= n`, cộng `min(last, n) - first + 1`, cập nhật `last * 10 + 9`?
- [ ] Đã thử `n = 1, k = 1`; `k = n`; `n = 10` (k = 2, 3); `n = 13, k = 2`; `n = 10^9, k = 10^9`?
- [ ] (C++/Java) `first`, `last` là `long`?
- [ ] Đúng signature `findKthNumber(self, n: int, k: int) -> int`, không còn `print` debug?